# Node prediction: what the models read about the grid (no training)

Four analyses of graphs and runs that already exist (added 2026-11-19, after the results; a prediction was
stated in advance only for analysis 4). They need `f1_graph_population.ipynb`'s outputs and the K-sweep checkpoints
(`RGCN_k_sweep.ipynb`, `graph_sage_k_sweep.ipynb`, whose K = 3 rows reuse the main runs' checkpoints).

1. **Hub degree.** Every node's input includes its standardised log degree (`f1_node_data.degree_feature`), and a
   `Position/<n>` hub's degree is the number of entries and sessions that hold that slot. The hub is one hop from
   its entries in every KG, so its degree can tell a model something about the grid before the rank literal is
   within reach. `hub_degree_rule_scores` scores each entry by that degree alone, next to the grid-only rule.
2. **Rank sensitivity by depth.** For grid slots 1-14 the hub degrees differ by less than 1%, so within that band
   the degree cannot separate slots and any ordering by slot must come from the rank itself (or from facts
   correlated with it). Each K-sweep test checkpoint is re-scored and its Spearman correlation with the grid slot is
   computed inside the band, split by whether the entry has a session position (the qualifying route to a rank).
3. **Predicate sharing in Pure SULO.** How many Ad-hoc properties each SULO predicate carries, and how many edges of
   the position edge's relation type a node has in the masked graph.
4. **Driver and team strength.** Within a grid slot and decade, does a model's score follow the driver's (or team's)
   share of point-scoring training entries? A model can identify a driver or team through its name literal, which
   is 2 hops from the entry in Ad-hoc and Hybrid and 4 in Pure SULO (behind the Role node and the name's
   information object). If Pure SULO's lower ROC-AUC comes from missing driver and team identity, its scores should
   follow these shares less than Ad-hoc's at K = 3 (names out of reach) and possibly catch up at K = 4.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "f1_population.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from collections import Counter

import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

from f1_node_data import (
    OUTCOME_PROPERTIES, LABEL_PROPERTY, _triples, build_kg_data, grid_rule_scores, hub_degree_rule_scores,
    load_population, masked_lines, race_entry_labels, stratified_split, train_point_rate, within_slot_spearman,
)
from f1_population import NS, SCHEMA_FILES, read_schema
from link_prediction.link_prediction_tools import session_position_entries
from node_prediction.gnn_tools import cleanup
from node_prediction.models import GraphSAGEModel, RGCNModel

NP_DIR = ROOT / "outputs" / "f1" / "node_prediction"
OUT_DIR = NP_DIR / "analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
KGS = ["adhoc", "hybrid_sulo", "pure_sulo"]
NAMES = {"adhoc": "Ad-hoc", "hybrid_sulo": "Hybrid", "pure_sulo": "Pure SULO"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

entities, assertions = load_population(ROOT / "data" / "f1")
labels = race_entry_labels(assertions)
split = stratified_split(labels, seed=42)
schemas = {kg: read_schema(str(ROOT / path)) for kg, path in SCHEMA_FILES.items()}
kg_data = {kg: build_kg_data(kg, entities, assertions, split, schemas[kg]) for kg in KGS}

/home/domin/miniconda3/envs/sulo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Hub degree

Each `Position/<n>` hub's degree in the masked graph (finish positions are masked, start and session positions stay),
then the two single-rule baselines per split. Both rules use only `hasStartPosition`; an entry without one is ranked
last by both.

In [2]:
rank_of = assertions.query("property == 'hasRank'").set_index("subject")["value"].astype(int)
hub_degree = pd.DataFrame({
    NAMES[kg]: {rank_of[hub]: int(torch.bincount(d.data.edge_index[1], minlength=d.data.num_nodes)[d.node_ids[f"<{hub}>"]])
                for hub in rank_of.index}
    for kg, d in kg_data.items()
}).sort_index().rename_axis("grid slot")
rho = spearmanr(hub_degree.index, hub_degree["Ad-hoc"]).statistic
print(f"Spearman(grid slot, hub degree) = {rho:.3f}")
# Typing and the rank's shape add the same number of edges to every hub, so the KGs differ by a constant.
assert all(hub_degree[NAMES[kg]].sub(hub_degree["Ad-hoc"]).nunique() == 1 for kg in KGS)
hub_degree.to_csv(OUT_DIR / "position_hub_degree.csv")
hub_degree.T

Spearman(grid slot, hub degree) = -0.937


grid slot,1,2,3,4,5,6,7,8,9,10,...,25,26,27,28,29,30,31,32,33,34
Ad-hoc,1802,1805,1810,1813,1804,1801,1811,1807,1813,1806,...,311,258,33,20,15,9,8,7,3,3
Hybrid,1803,1806,1811,1814,1805,1802,1812,1808,1814,1807,...,312,259,34,21,16,10,9,8,4,4
Pure SULO,1803,1806,1811,1814,1805,1802,1812,1808,1814,1807,...,312,259,34,21,16,10,9,8,4,4


In [3]:
BAND = range(1, 15)   # grid slots whose hub degrees differ by < 1% (checked below)
band_degree = hub_degree.loc[list(BAND), "Ad-hoc"]
assert band_degree.max() / band_degree.min() < 1.01, band_degree

grid = -grid_rule_scores(assertions, labels.index)
in_band = grid.isin(BAND)
rules = {}
for name, iris in zip(["train", "val", "test"], split):
    y = labels.loc[iris]
    by_kg = {kg: hub_degree_rule_scores(d, assertions, iris) for kg, d in kg_data.items()}
    degree_auc = {kg: roc_auc_score(y, s) for kg, s in by_kg.items()}
    assert len({round(v, 12) for v in degree_auc.values()}) == 1, degree_auc   # same order in every KG
    band = in_band.loc[iris]
    rules[name] = {"grid_rule_roc_auc": roc_auc_score(y, -grid.loc[iris]),
                   "hub_degree_rule_roc_auc": degree_auc["adhoc"],
                   "band_grid_rule_roc_auc": roc_auc_score(y[band], -grid.loc[iris][band]),
                   "band_hub_degree_rule_roc_auc": roc_auc_score(y[band], by_kg["adhoc"][band]),
                   "band_entries": int(band.sum())}
rules = pd.DataFrame(rules).T.rename_axis("split")
rules.to_csv(NP_DIR / "hub_degree_rule_baseline.csv")
rules.round(4)

,grid_rule_roc_auc,hub_degree_rule_roc_auc,band_grid_rule_roc_auc,band_hub_degree_rule_roc_auc,band_entries
split,,,,,
train,0.8176,0.7444,0.7012,0.5236,12897.0
val,0.8145,0.7438,0.7018,0.5280,1601.0
test,0.8352,0.7567,0.7240,0.5250,1575.0


## 2. Rank sensitivity by depth

The K-sweep's test checkpoints (retrained on train + val, 5 seeds), rebuilt from each K's configuration in
`k_sweep_results.csv`. Inside the band, `rho` is the Spearman correlation between the model's score and the grid
slot (sign flipped, so a model that ranks the front of the grid higher has rho > 0). Pooled `rho` mixes eras, so it is
lower than both session groups.

In [4]:
# Must equal the K-sweep notebooks' constants; the state dict would not load otherwise.
FEATURE_DIM = 256
DROPOUT = {"rgcn": 0.2, "graphsage": 0.5}   # inactive in eval mode
ENCODERS = {"rgcn": "R-GCN", "graphsage": "GraphSAGE"}

test = list(split[2])
y_test = labels.loc[test].to_numpy()
slot = grid.loc[test]
band = slot.isin(BAND).to_numpy()
with_session = np.array([e in session_position_entries(assertions) for e in test])


def checkpoint(enc, kg, k, seed):
    """K = 3 is the tuned depth: the K-sweep reused the main run's final checkpoints."""
    if k == 3:
        return NP_DIR / enc / kg / f"{kg}_final_unified_eval_seed{seed}.pt"
    return NP_DIR / "k_sweep" / enc / kg / f"{kg}_K{k}_final_unified_eval_seed{seed}.pt"


def score_checkpoint(enc, d, row, path):
    """Test logits of one saved checkpoint, from one eval-mode full-batch forward pass."""
    k = int(row["K"])
    hidden = [int(row["hidden_dim"])] * (k - 1) if k > 1 else []   # K = 1 logs hidden_dim None
    if enc == "rgcn":
        bases = None if pd.isna(row["num_bases"]) else int(row["num_bases"])
        model = RGCNModel(len(d.vocabulary), FEATURE_DIM, hidden, num_relations=d.num_relations, num_bases=bases,
                          dropout=DROPOUT[enc])
    else:
        model = GraphSAGEModel(len(d.vocabulary), FEATURE_DIM, hidden, dropout=DROPOUT[enc])
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=False)["model_state_dict"])
    model = model.to(DEVICE).eval()
    graph = d.data.clone().to(DEVICE)
    with torch.no_grad():
        scores = model(graph)[d.test.to(DEVICE)].squeeze(1).cpu().numpy()
    model.cpu()
    del model, graph
    cleanup()
    return scores


def rho(scores, mask):
    return spearmanr(scores[mask], -slot.to_numpy()[mask]).statistic


per_seed, test_scores = [], {}   # test_scores: (encoder, kg, K, seed) -> logits, reused in section 4
for enc in ENCODERS:
    sweep = pd.read_csv(NP_DIR / "k_sweep" / enc / "k_sweep_results.csv")
    for kg, d in kg_data.items():
        for _, row in sweep.query("kg == @kg and phase == 'tune'").iterrows():
            finals = sweep.query("kg == @kg and K == @row.K and phase == 'final'")
            for _, final in finals.iterrows():
                scores = score_checkpoint(enc, d, row, checkpoint(enc, kg, int(row["K"]), int(final["seed"])))
                test_scores[(enc, kg, int(row["K"]), int(final["seed"]))] = scores
                per_seed.append({"encoder": enc, "kg": kg, "K": int(row["K"]), "seed": int(final["seed"]),
                                 "test_roc_auc": roc_auc_score(y_test, scores),
                                 "csv test_roc_auc": final["test_roc_auc"],
                                 "band_roc_auc": roc_auc_score(y_test[band], scores[band]),
                                 "band_rho": rho(scores, band),
                                 "band_rho_with_session": rho(scores, band & with_session),
                                 "band_rho_without_session": rho(scores, band & ~with_session)})
per_seed = pd.DataFrame(per_seed)
# Re-scoring a checkpoint must give the test ROC-AUC its run logged (GPU scatter noise aside).
assert np.allclose(per_seed["test_roc_auc"], per_seed["csv test_roc_auc"], atol=2e-3), "re-scored ROC-AUC differs"
per_seed.to_csv(OUT_DIR / "rank_sensitivity_per_seed.csv", index=False)
print(f"band: {band.sum()} test entries ({(band & with_session).sum()} with a session position); "
      f"grid rule ROC-AUC {roc_auc_score(y_test[band], -slot[band]):.3f}, "
      f"hub-degree rule {rules.loc['test', 'band_hub_degree_rule_roc_auc']:.3f}")

metrics = ["test_roc_auc", "band_roc_auc", "band_rho", "band_rho_with_session", "band_rho_without_session"]
sensitivity = per_seed.groupby(["encoder", "kg", "K"], sort=False)[metrics].agg(["mean", "std"])
sensitivity.round(4).to_csv(OUT_DIR / "rank_sensitivity.csv")
sensitivity.round(3)

band: 1575 test entries (722 with a session position); grid rule ROC-AUC 0.724, hub-degree rule 0.525


test_roc_auc        band_roc_auc        band_rho  \
                                mean    std         mean    std     mean   
encoder   kg          K                                                    
rgcn      adhoc       1        0.783  0.005        0.706  0.001    0.332   
                      2        0.869  0.001        0.786  0.001    0.757   
                      3        0.868  0.002        0.784  0.002    0.771   
                      4        0.869  0.001        0.784  0.001    0.734   
          hybrid_sulo 1        0.789  0.001        0.708  0.001    0.333   
                      2        0.827  0.013        0.751  0.011    0.456   
                      3        0.869  0.001        0.786  0.001    0.758   
                      4        0.866  0.002        0.782  0.001    0.753   
          pure_sulo   1        0.684  0.015        0.607  0.009    0.033   
                      2        0.761  0.007        0.701  0.003    0.244   
                      3        0.855  0.002        0.766  0.006    0.911   
                      4        0.855  0.002        0.767  0.003    0.895   
graphsage adhoc       1        0.728  0.000        0.687  0.000    0.202   
                      2        0.821  0.003        0.739  0.003    0.417   
                      3        0.863  0.002        0.780  0.002    0.715   
                      4        0.862  0.001        0.777  0.001    0.709   
          hybrid_sulo 1        0.689  0.087        0.658  0.064    0.133   
                      2        0.815  0.004        0.735  0.004    0.408   
                      3        0.859  0.008        0.777  0.009    0.673   
                      4        0.847  0.013        0.764  0.013    0.561   
          pure_sulo   1        0.641  0.053        0.614  0.007    0.029   
                      2        0.705  0.008        0.677  0.005    0.162   
                      3        0.782  0.002        0.709  0.001    0.305   
                      4        0.783  0.004        0.710  0.002    0.305   

                               band_rho_with_session         \
                           std                  mean    std   
encoder   kg          K                                       
rgcn      adhoc       1  0.001                 0.383  0.001   
                      2  0.008                 0.823  0.012   
                      3  0.016                 0.875  0.022   
                      4  0.013                 0.830  0.016   
          hybrid_sulo 1  0.001                 0.380  0.001   
                      2  0.032                 0.525  0.015   
                      3  0.029                 0.825  0.021   
                      4  0.010                 0.893  0.013   
          pure_sulo   1  0.002                -0.028  0.051   
                      2  0.012                 0.382  0.007   
                      3  0.032                 0.975  0.005   
                      4  0.015                 0.976  0.006   
graphsage adhoc       1  0.000                 0.314  0.000   
                      2  0.012                 0.477  0.013   
                      3  0.013                 0.783  0.012   
                      4  0.010                 0.763  0.014   
          hybrid_sulo 1  0.155                 0.188  0.278   
                      2  0.016                 0.461  0.016   
                      3  0.077                 0.743  0.082   
                      4  0.081                 0.617  0.108   
          pure_sulo   1  0.005                -0.008  0.052   
                      2  0.014                 0.289  0.012   
                      3  0.012                 0.403  0.001   
                      4  0.021                 0.399  0.009   

                        band_rho_without_session         
                                            mean    std  
encoder   kg          K                                  
rgcn      adhoc       1                    0.354  0.004  
                      2              

## 3. Predicate sharing in Pure SULO

Per SULO predicate, the Ad-hoc properties that Pure SULO writes with it (from the schema), then, in the masked
node-prediction graphs, the mean number of edges of the position edge's relation type per node.

In [5]:
carried = pd.Series({prop: shape.predicate.rsplit("/", 1)[-1] for prop, shape in schemas["pure_sulo"].properties.items()})
masked_props = OUTCOME_PROPERTIES | {LABEL_PROPERTY}
sharing = pd.DataFrame({
    "properties": carried.groupby(carried).size(),
    "after the NC mask": carried[~carried.index.isin(masked_props)].groupby(carried[~carried.index.isin(masked_props)]).size(),
    "which": carried.groupby(carried).apply(lambda s: ", ".join(sorted(s.index))),
}).fillna(0).astype({"after the NC mask": int}).sort_values("properties", ascending=False)
sharing

,properties,after the NC mask,which
hasFeature,15,11,"hasBestLapTime, hasCountry, hasFinishPosition,..."
isReferredToIn,6,6,"hasCircuitName, hasDriverName, hasRaceName, ha..."
atTime,3,3,"hasDateOfBirth, hasRaceDate, hasYear"
precedes,2,2,"precedesRace, precedesSeason"
hasParticipant,2,2,"hasDriver, hasTeam"
hasPart,1,1,hasRaceEntry
hasItem,1,1,hasTeamMember
isPartOf,1,1,partOfSeason
isIn,1,1,heldAtCircuit


In [6]:
class_of = dict(zip(entities["iri"], entities["class"]))
# (node class, Ad-hoc property of the position edge); the edge leaves the subject.
POSITION_EDGES = [("RaceEntry", "hasStartPosition"), ("SessionResult", "hasPosition")]
fan_out = {}
for kg in ["hybrid_sulo", "pure_sulo"]:
    triples = list(_triples(masked_lines(entities, assertions, schemas[kg])))
    per_subject = Counter((s, p) for s, p, o in triples)
    for cls, prop in POSITION_EDGES:
        predicate = f"<{schemas[kg].properties[prop].predicate}>"
        holders = {s for s, p, o in triples if p == predicate and class_of.get(s.strip("<>")) == cls
                   and o.strip("<>").startswith(NS + "Position/")}
        fan_out[(NAMES[kg], f"{cls} ({prop})")] = np.mean([per_subject[(s, predicate)] for s in holders])
pd.Series(fan_out).unstack(0).round(2).rename_axis("node (position edge)")

,Hybrid,Pure SULO
node (position edge),,
RaceEntry (hasStartPosition),1.0,1.00
SessionResult (hasPosition),1.0,3.21


## 4. Driver and team strength

`train_point_rate`: the share of the driver's (team's) other training entries that scored points, from the labels
of the training split only. `within_slot_spearman`: the Spearman correlation inside each stratum (grid slot ×
decade, so neither the slot nor the era's points rules count), weighted by the stratum's size. The first rows are
reference values: how strongly the label itself follows each share within the strata. Test entries with a grid slot.

In [7]:
from link_prediction.link_prediction_tools import evidence_hops

hops = pd.DataFrame({NAMES[kg]: {"driver name": evidence_hops(schemas[kg], ("hasDriver", "hasDriverName")),
                                 "team name": evidence_hops(schemas[kg], ("hasTeam", "hasTeamName"))} for kg in KGS})
display(hops.rename_axis("hops from the entry"))

value = lambda prop: assertions.query("property == @prop").set_index("subject")["value"]
entry_race = assertions.query("property == 'hasRaceEntry'").set_index("value")["subject"]
year = pd.Series(test, index=test).map(entry_race).map(value("partOfSeason")).map(value("hasYear")).astype(int)
on_grid = slot.isin(rank_of.values).to_numpy()
stratum = (slot.astype(int).astype(str) + "_" + (year // 10 * 10).astype(str)).to_numpy()[on_grid]
rate = {name: train_point_rate(assertions, labels, split[0], prop).loc[test].to_numpy()[on_grid]
        for name, prop in [("driver", "hasDriver"), ("team", "hasTeam")]}

strength = [{"encoder": "label", "kg": "", "K": 0, "seed": 0,
             **{f"rho {name}": within_slot_spearman(y_test[on_grid], r, stratum) for name, r in rate.items()},
             "rho label": 1.0}]
for (enc, kg, k, seed), scores in test_scores.items():
    s = scores[on_grid]
    strength.append({"encoder": enc, "kg": kg, "K": k, "seed": seed,
                     **{f"rho {name}": within_slot_spearman(s, r, stratum) for name, r in rate.items()},
                     "rho label": within_slot_spearman(s, y_test[on_grid], stratum)})
strength = pd.DataFrame(strength)
strength.to_csv(OUT_DIR / "driver_team_strength_per_seed.csv", index=False)
strength_summary = strength.groupby(["encoder", "kg", "K"], sort=False)[["rho driver", "rho team", "rho label"]].agg(["mean", "std"])
strength_summary.round(4).to_csv(OUT_DIR / "driver_team_strength.csv")
print(f"{on_grid.sum()} test entries with a grid slot in {len(set(stratum))} strata")
strength_summary.round(3)

,Ad-hoc,Hybrid,Pure SULO
hops from the entry,,,
driver name,2,2,4
team name,2,2,4


2553 test entries with a grid slot in 199 strata


rho driver        rho team        rho label       
                              mean    std     mean    std      mean    std
encoder   kg          K                                                   
label                 0      0.168    NaN    0.176    NaN     1.000    NaN
rgcn      adhoc       1      0.510  0.009    0.401  0.007     0.146  0.002
                      2      0.590  0.077    0.664  0.087     0.189  0.005
                      3      0.482  0.073    0.703  0.038     0.180  0.014
                      4      0.672  0.026    0.690  0.032     0.184  0.005
          hybrid_sulo 1      0.507  0.004    0.389  0.005     0.143  0.002
                      2      0.576  0.166    0.736  0.109     0.186  0.012
                      3      0.529  0.068    0.752  0.034     0.186  0.007
                      4      0.406  0.066    0.768  0.040     0.177  0.007
          pure_sulo   1     -0.013  0.017   -0.012  0.020     0.006  0.007
                      2      0.397  0.028    0.380  0.008     0.117  0.007
                      3      0.372  0.035    0.339  0.047     0.109  0.013
                      4      0.312  0.043    0.350  0.039     0.090  0.029
graphsage adhoc       1      0.382  0.000    0.345  0.000     0.101  0.000
                      2      0.785  0.024    0.396  0.036     0.186  0.006
                      3      0.739  0.013    0.559  0.018     0.195  0.002
                      4      0.739  0.012    0.573  0.016     0.190  0.005
          hybrid_sulo 1      0.227  0.324    0.206  0.306     0.059  0.098
                      2      0.814  0.019    0.373  0.027     0.179  0.006
                      3      0.719  0.013    0.554  0.016     0.191  0.005
                      4      0.756  0.008    0.554  0.030     0.191  0.008
          pure_sulo   1      0.007  0.021   -0.004  0.027    -0.009  0.010
                      2      0.273  0.026    0.231  0.024     0.076  0.012
                      3      0.543  0.005    0.367  0.016     0.153  0.005
                      4      0.547  0.010    0.374  0.024     0.151  0.004